In [ ]:
import numpy as np
from keras.datasets import mnist
from sklearn.model_selection import KFold
from sklearn.metrics import pairwise_distances_argmin_min
from sklearn.preprocessing import normalize
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

Custom distance calculation function for L2 (Euclidean), L1 (Manhattan), and Cosine distances.

In [ ]:
def computeDistance(x, centroids, metric):
    # classic euclidean distance function - it gives x - cent
    if metric == "l2":
        return np.linalg.norm(x - centroids, axis=1)  
    # manhattan distance - it so bad bcs image data comes with a lot of zero...
    elif metric == "l1":
        return np.sum(np.abs(x - centroids), axis=1) 
    # cosine distance - vector direction depends
    elif metric == "cosine":
        x_norm = normalize(x.reshape(1, -1))
        centroids_norm = normalize(centroids)
        return 1 - np.dot(centroids_norm, x_norm.T).reshape(-1)  # Cosine mesafesi
    return None

# performs k-means clustering with custom distance metric and centroid update it depends on l1-l2-cosine in if checks also i check emptiness
def kMeansCustom(xTrain, k, metric="l2", maxIter=100):
    # maxIter = 200 # çok farklı sonuc verdi
    
    np.random.seed(42)
    indices = np.random.choice(xTrain.shape[0], k, replace=False)
    centroids = xTrain[indices]

    for _ in range(maxIter):# Her noktayı en yakın merkez noktasına ata
        labels = np.array([np.argmin(computeDistance(x, centroids, metric)) for x in xTrain])

        new_centroids = np.zeros_like(centroids)
        for i in range(k):
            cluster_points = xTrain[labels == i]

            if len(cluster_points) == 0:  # Boş küme kontrolü
                new_centroids[i] = centroids[i]
            else:
                if metric == "l2": # L2 için aritmetik ortalama
                    new_centroids[i] = cluster_points.mean(axis=0)
                elif metric == "l1": # L1 için medyan
                    try:
                        #print("l1"+ str(i))
                        new_centroids[i] = np.median(cluster_points, axis=0)
                    except: # pcadan sonra bazıları boş array oluyo sıkıntı oluyordu
                        new_centroids[i] = centroids[i]
                elif metric == "cosine": # Cosine için normalize edilmiş toplam vektör
                    #print("cosine"+ str(i))
                    cluster_sum = np.sum(cluster_points, axis=0)
                    # abi burda division error felan verio
                    # norm = np.sqrt(np.sum(cluster_sum**2))
                    norm = np.linalg.norm(cluster_sum)
                    if norm > 1e-6:
                        new_centroids[i] = cluster_sum / norm
                    else:
                        # Çok küçük ya da sıfır vektör oluştu, centroid sabit kalsın
                        new_centroids[i] = centroids[i]
                    # new_centroids[i] = cluster_sum / norm if norm > 0 else centroids[i] # ama pcada sıkıntı çıkardı... githubda böyle bi sayılarla denenmiş

        # Yakınsama kontrolü
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids

    return centroids, labels

In [ ]:
def getClusterLabelMatrix(trueLabels, clusterLabels, numClusters=10):
    matrix = np.zeros((10, numClusters), dtype=int)

    for true, cluster in zip(trueLabels, clusterLabels):
        matrix[true, cluster] += 1
    
    # get how mant instances for everty true label in given cluestr
    return matrix

# nij ntablosunda hangi satırlar kaç kere recc edildiğini gösterir ve
# oraya assign ederiz bu şekilde yakına gitmiş olur
def assignLabelsToClusters(countMatrix):
    clusterLabels = [-1] * countMatrix.shape[1]
    labelUsed = set()
    clusterUsed = set()
    
    flat = []
    for label in range(countMatrix.shape[0]):
        for cluster in range(countMatrix.shape[1]):
            flat.append((countMatrix[label, cluster], label, cluster))
    
    flat.sort(reverse=True)  # desc seq

    for count, label, cluster in flat:
        if cluster not in clusterUsed and label not in labelUsed:
            clusterLabels[cluster] = label
            clusterUsed.add(cluster)
            labelUsed.add(label)

    # sometimes it is empty so assing -1 
    for cluster in range(countMatrix.shape[1]):
        if clusterLabels[cluster] == -1:
            clusterLabels[cluster] = -1  # may i can find best value how?

    return clusterLabels  

### Cluster Label Assignment Logic

After applying k-means clustering, each data point in the training set is assigned to one of the k clusters. Since k=10, the goal is to match each cluster with the most likely digit label (0–9).

To do this, I built a 10x10 table where each cell (i, j) represents the number of training samples with true label j assigned to cluster i. Then, I followed this labeling strategy:

1. I looked for the highest value in the table. The corresponding cluster is labeled with that digit.
2. After assigning one label to a cluster, I marked both the cluster and the label as “used”.
3. I continued selecting the next highest value in the table, skipping already labeled clusters or already assigned labels.
4. This process continued until all 10 clusters had unique labels.

This way, each cluster gets a label based on the most dominant class in it, and each digit is assigned only once. This helps evaluate accuracy by comparing predicted cluster labels with true labels.


In [ ]:
def evaluateTrainAccuracy(trueLabels, clusterAssignments, clusterToLabelMap): # evaluate the acc of the clustering
    predictedLabels = [clusterToLabelMap[cluster] for cluster in clusterAssignments]
    acc = accuracy_score(trueLabels, predictedLabels)
    conf = confusion_matrix(trueLabels, predictedLabels, labels=list(range(10)))
    #print("Confusion Matrix:")
    #print(conf)
    #print("Accuracy:")
    #print(acc)
    return acc, conf

#### Result of acc:
- L2 norm performs better than L1 due to smoother centroid update with mean values
- Cosine distance is not good i though vectorized after i tried the vectorized it goes up%20.
- L1 is okey with outliers but if numbers change it goes down as expected

In [ ]:
def predictTestLabels(xTest, centroids, clusterToLabel, metric="l2"):
    predictedLabels = []
  
    # find closest point in area
    for x in xTest:
        distances = computeDistance(x, centroids, metric)
        closestCluster = np.argmin(distances)
        predictedLabels.append(clusterToLabel[closestCluster])

    return np.array(predictedLabels)

def evaluateTestAccuracy(yTest, yPredicted):
    acc = accuracy_score(yTest, yPredicted)
    conf = confusion_matrix(yTest, yPredicted, labels=list(range(10)))
    return acc, conf

In [ ]:
def summarizeResults(results): # general result print section
    summary = {}

    for res in results:
        metric = res["metric"]

        if metric not in summary:
            summary[metric] = {"train": [], "test": []}
            
        summary[metric]["train"].append(res["trainAccuracy"])
        summary[metric]["test"].append(res["testAccuracy"])
    
    for metric in summary:
        trainMean = np.mean(summary[metric]["train"])
        testMean = np.mean(summary[metric]["test"])
        print(f"{metric.upper()} → Train Acc: {trainMean:.4f} | Test Acc: {testMean:.4f}")

In [ ]:
def plotConfusionMatrix(confMatrix, title="Confusion Matrix"): # plot conf matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(confMatrix, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def plotAccuracyComparison(results): # acc compariton plting
    metrics = ["l2", "l1", "cosine"]
    trainAvgs = []
    testAvgs = []

    # getting datas from results to plotting
    for metric in metrics:
        trainAccs = [r["trainAccuracy"] for r in results if r["metric"] == metric]
        testAccs = [r["testAccuracy"] for r in results if r["metric"] == metric]
        trainAvgs.append(np.mean(trainAccs))
        testAvgs.append(np.mean(testAccs))

    x = np.arange(len(metrics))
    width = 0.35

    fig, ax = plt.subplots()
    ax.bar(x - width/2, trainAvgs, width, label='Train Acc')
    ax.bar(x + width/2, testAvgs, width, label='Test Acc')

    ax.set_ylabel('Accuracy')
    ax.set_title('Average Accuracy by Metric')
    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
(x, y), _ = mnist.load_data() # get data and reshape and run hmoework
x = x.reshape((x.shape[0], -1)) / 255.0

# in testing get 1k data it takes 73minutes ...
x = x[:1000]
y = y[:1000]


# create fols 5 for cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(x, y))

results = []
foldResults = {
    "l2": [],
    "l1": [],
    "cosine": []
}

# run cross validation
for i, (trainIdx, testIdx) in enumerate(folds):
    xTrain, yTrain = x[trainIdx], y[trainIdx]
    xTest, yTest = x[testIdx], y[testIdx]

    for metric in ["l2", "l1", "cosine"]:
        centroids, clusterAssignments = kMeansCustom(xTrain, k=10, metric=metric)
        
        countMatrix = getClusterLabelMatrix(yTrain, clusterAssignments)
        clusterToLabel = assignLabelsToClusters(countMatrix)

        trainAcc, trainConf = evaluateTrainAccuracy(yTrain, clusterAssignments, clusterToLabel)
        yPredTest = predictTestLabels(xTest, centroids, clusterToLabel, metric=metric)
        testAcc, testConf = evaluateTestAccuracy(yTest, yPredTest)

        res = {
            "fold": i + 1,
            "trainAccuracy": trainAcc,
            "testAccuracy": testAcc
        }
        foldResults[metric].append(res)

        results.append({
            "trainAccuracy": trainAcc,
            "trainConfusion": trainConf,
            "testAccuracy": testAcc,
            "testConfusion": testConf,
            "metric": metric
        })

        plotConfusionMatrix(testConf, title=f"{metric.upper()} Test Confusion Matrix (Fold {i+1})")

# fold by fold print datas
print("\nFOLD-BY-FOLD ACCURACY REPORT\n" + "-"*35)
for metric in ["l2", "l1", "cosine"]:

    print(f"\nMetric:" + metric.upper())

    for res in foldResults[metric]:
        print(f"Fold {res['fold']}: Train Acc = {res['trainAccuracy']:.4f}, Test Acc = {res['testAccuracy']:.4f}")

    avgTrain = sum(r["trainAccuracy"] for r in foldResults[metric]) / 5
    avgTest = sum(r["testAccuracy"] for r in foldResults[metric]) / 5

    print(f"Avg Train Accuracy: {avgTrain:.4f}")
    print(f"Avg Test Accuracy : {avgTest:.4f}")

# summart of metrics
df = pd.DataFrame(results)
for metric in ["l2", "l1", "cosine"]:
    metricDf = df[df["metric"] == metric]
    print(f"\n{metric.upper()} Results:")
    print("Average Train Accuracy:", metricDf["trainAccuracy"].mean().round(4))
    print("Average Test Accuracy:", metricDf["testAccuracy"].mean().round(4))

summarizeResults(results)
plotAccuracyComparison(results)



- I tried k=25 at first, but it clearly overfit.
- With 1000 samples and cosine distance runtime was getting close to 3-5 mins why there is so difference i did not get it.
- Normalizing the data before cosine seemed unnecessary move - no need belhanda — results were similar.

Might test with PCA next time to reduce runtime.

In [ ]:
from sklearn.decomposition import PCA
 
resultsPca = []
nComponents = 50  # PCA bileşen sayısı
for trainIdx, testIdx in folds:
    xTrain, yTrain = x[trainIdx], y[trainIdx]
    xTest, yTest = x[testIdx], y[testIdx]

    for metric in ["l2", "l1", "cosine"]:
        # PCA ile boyut indir
        pca = PCA(n_components=nComponents, random_state=42)
        xTrainReduced = pca.fit_transform(xTrain)
        xTestReduced = pca.transform(xTest)

        centroids, clusterAssignments = kMeansCustom(xTrainReduced, k=10, metric=metric)

        countMatrix = getClusterLabelMatrix(yTrain, clusterAssignments)
        clusterToLabel = assignLabelsToClusters(countMatrix)

        # eğitim verisiyle acc
        trainAccuracy, trainConfusion = evaluateTrainAccuracy(yTrain, clusterAssignments, clusterToLabel)

        # test verisi
        predictedTestLabels = predictTestLabels(xTestReduced, centroids, clusterToLabel, metric=metric)
        testAccuracy = accuracy_score(yTest, predictedTestLabels)
        testConfusion = confusion_matrix(yTest, predictedTestLabels, labels=list(range(10)))

        # it is same as without pca version up
        res = {
            "metric": metric,
            "trainAccuracy": trainAccuracy,
            "trainConfusion": trainConfusion,
            "testAccuracy": testAccuracy,
            "testConfusion": testConfusion
        }
        resultsPca.append(res)
        plotConfusionMatrix(res["testConfusion"], title=f"{res['metric'].upper()} Test Set Confusion Matrix (PCA)")

# PCA sonuçlarını özetle
summarizeResults(resultsPca)
plotAccuracyComparison(resultsPca)

### Results and Observations

| Distance Metric | Train Accuracy (Raw) | Test Accuracy (Raw) | Train Accuracy (PCA) | Test Accuracy (PCA) |
|------------------|-----------------------|------------------------|------------------------|------------------------|
| **L2 (Euclidean)** | 0.5033 | 0.5140 | 0.4827 | 0.4780 |
| **L1 (Manhattan)** | 0.4615 | 0.4580 | 0.4647 | 0.4680 |
| **Cosine**         | 0.5113 | 0.5030 | 0.4885 | 0.4900 |

#### Evaluation

- Cosine had the highest training accuracy on raw data. L2 had slightly better test performance.
- After PCA, the results became more balanced across all three metrics.
- L1 showed the lowest performance both before and after PCA.
- The results are acceptable for an unsupervised task on high-dimensional image data.

#### Conclusion

Cosine distance captures directional similarity and worked well on raw pixel space. L2 gave more stable results on test data. PCA reduced dimensionality and made metric differences less significant. The choice of distance metric clearly affects the clustering outcome.


### Results

#### 1. Without PCA

The clustering was applied to the raw MNIST data using three distance metrics. The accuracy scores averaged over 5-fold cross-validation are summarized below:

| Metric      | Train Accuracy | Test Accuracy |
|-------------|----------------|----------------|
| L2          | 0.5033         | 0.5140         |
| L1          | 0.4615         | 0.4580         |
| Cosine      | 0.5113         | 0.5030         |

Cosine distance performed best in training, while L2 slightly outperformed in the test set. L1 distance gave the lowest results overall.

#### 2. With PCA

When PCA was applied before clustering, dimensionality was reduced while keeping most of the variance. The new accuracy values are below:

| Metric      | Train Accuracy | Test Accuracy |
|-------------|----------------|----------------|
| L2          | 0.4827         | 0.4780         |
| L1          | 0.4647         | 0.4680         |
| Cosine      | 0.4885         | 0.4900         |

With PCA, the differences between metrics were smaller. Cosine was still the best in most cases, though overall accuracy was slightly reduced compared to the raw data.

#### 3. Comparison

| Metric      | Best on Train | Best on Test |
|-------------|----------------|----------------|
| Raw         | Cosine         | L2             |
| With PCA    | Cosine         | Cosine         |

PCA appears to balance the metrics but may slightly reduce performance. Cosine was the most stable overall.
